# Student Engagement Detection via Facial Emotion Recognition

This notebook builds a CNN-based emotion classifier on the **FER+** dataset to detect student engagement from facial expressions.

**Emotion classes:** Angry, Disgust, Fear, Happy, Neutral, Sad, Surprise

## 0. Imports & Setup

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.utils import to_categorical

from PIL import Image
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_curve, auc, accuracy_score,
    precision_score, recall_score, f1_score
)
from sklearn.preprocessing import label_binarize
from itertools import cycle

print('TensorFlow version:', tf.__version__)
print('GPU available:', tf.config.list_physical_devices('GPU'))

# Reproducibility
np.random.seed(42)
tf.random.set_seed(42)

## 1. Data Acquisition

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────
BASE_DIR   = Path('..')
TRAIN_DIR  = BASE_DIR / 'data' / 'train'
TEST_DIR   = BASE_DIR / 'data' / 'test'
MODEL_DIR  = BASE_DIR / 'models'
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# ── Fixed constants ────────────────────────────────────────────────────────
IMG_SIZE   = 48
BATCH_SIZE = 64
EPOCHS     = 50

print('Train dir:', TRAIN_DIR)
print('Test  dir:', TEST_DIR)

# Discover classes from the actual folder structure
train_classes = sorted([
    d for d in os.listdir(TRAIN_DIR)
    if os.path.isdir(TRAIN_DIR / d)
])
NUM_CLASSES = len(train_classes)
print('Classes found:', train_classes)
print('Number of classes:', NUM_CLASSES)

In [ ]:
# Count images per class
def count_images(data_dir):
    counts = {}
    for cls in sorted(os.listdir(data_dir)):
        cls_path = Path(data_dir) / cls
        if cls_path.is_dir():
            imgs = [f for f in os.listdir(cls_path)
                    if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
            counts[cls] = len(imgs)
    return counts

train_counts = count_images(TRAIN_DIR)
test_counts  = count_images(TEST_DIR)

print('Train class distribution:')
for cls, cnt in train_counts.items():
    print(f'  {cls}: {cnt}')
print(f'  TOTAL: {sum(train_counts.values())}')

print('\nTest class distribution:')
for cls, cnt in test_counts.items():
    print(f'  {cls}: {cnt}')
print(f'  TOTAL: {sum(test_counts.values())}')

## 2. Data Preprocessing & Augmentation

In [ ]:
# ── Training generator: augmentation + normalisation ───────────────────────
train_datagen = ImageDataGenerator(
    rescale=1.0 / 255.0,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest',
    validation_split=0.15
)

# ── Validation / test generator: normalisation only ───────────────────────
test_datagen = ImageDataGenerator(rescale=1.0 / 255.0)

# ── Generators ─────────────────────────────────────────────────────────────
train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    color_mode='grayscale',
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    shuffle=True,
    seed=42
)

val_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    color_mode='grayscale',
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    shuffle=False,
    seed=42
)

test_generator = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    color_mode='grayscale',
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

# Use the generator's class_indices as the source of truth
CLASS_NAMES = [cls for cls, _ in
               sorted(train_generator.class_indices.items(), key=lambda x: x[1])]
NUM_CLASSES = len(CLASS_NAMES)
print('Class indices:', train_generator.class_indices)
print('Ordered class names:', CLASS_NAMES)

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# ── 3a. Class distribution bar chart ──────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = sns.color_palette('Set2', len(train_counts))

axes[0].bar(train_counts.keys(), train_counts.values(), color=colors)
axes[0].set_title('Training Set — Class Distribution', fontsize=13)
axes[0].set_xlabel('Emotion')
axes[0].set_ylabel('Number of Images')
axes[0].tick_params(axis='x', rotation=45)

axes[1].bar(test_counts.keys(), test_counts.values(), color=colors)
axes[1].set_title('Test Set — Class Distribution', fontsize=13)
axes[1].set_xlabel('Emotion')
axes[1].set_ylabel('Number of Images')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('class_distribution.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# ── 3b. Sample images from each class ─────────────────────────────────────
n_cols = 4
n_rows = (NUM_CLASSES + n_cols - 1) // n_cols  # ceiling division
fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 4 * n_rows))
axes = axes.flatten()

for idx, cls in enumerate(sorted(os.listdir(TRAIN_DIR))):
    cls_path = TRAIN_DIR / cls
    if not cls_path.is_dir():
        continue
    imgs = [f for f in os.listdir(cls_path)
            if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    if not imgs:
        continue
    img = Image.open(cls_path / imgs[0]).convert('L').resize((IMG_SIZE, IMG_SIZE))
    axes[idx].imshow(np.array(img), cmap='gray')
    axes[idx].set_title(cls, fontsize=11)
    axes[idx].axis('off')

# Hide any unused subplots
for ax in axes[NUM_CLASSES:]:
    ax.axis('off')

plt.suptitle('Sample Images per Emotion Class', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('sample_images.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# ── 3c. Pixel intensity distribution ──────────────────────────────────────
pixel_values = []
for cls in sorted(os.listdir(TRAIN_DIR)):
    cls_path = TRAIN_DIR / cls
    if not cls_path.is_dir():
        continue
    imgs = [f for f in os.listdir(cls_path)
            if f.lower().endswith(('.png', '.jpg', '.jpeg'))][:50]
    for img_name in imgs:
        img = np.array(
            Image.open(cls_path / img_name).convert('L').resize((IMG_SIZE, IMG_SIZE))
        )
        pixel_values.extend(img.flatten().tolist())

plt.figure(figsize=(10, 4))
plt.hist(pixel_values, bins=64, color='steelblue', edgecolor='white', alpha=0.85)
plt.title('Pixel Intensity Distribution (sample of training images)', fontsize=13)
plt.xlabel('Pixel Value (0–255)')
plt.ylabel('Frequency')
plt.tight_layout()
plt.savefig('pixel_distribution.png', dpi=100, bbox_inches='tight')
plt.show()

print(f'Mean pixel value : {np.mean(pixel_values):.2f}')
print(f'Std  pixel value : {np.std(pixel_values):.2f}')

## 4. Model Creation

In [ ]:
def build_emotion_cnn(input_shape=(48, 48, 1), num_classes=7):
    """CNN with 3 convolutional blocks + dense head for emotion classification."""

    model = keras.Sequential([
        # ── Block 1 ─────────────────────────────────────────────────────
        layers.Conv2D(64, (3, 3), padding='same', activation='relu',
                      input_shape=input_shape),
        layers.BatchNormalization(),
        layers.Conv2D(64, (3, 3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2, 2),
        layers.Dropout(0.25),

        # ── Block 2 ─────────────────────────────────────────────────────
        layers.Conv2D(128, (3, 3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.Conv2D(128, (3, 3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2, 2),
        layers.Dropout(0.25),

        # ── Block 3 ─────────────────────────────────────────────────────
        layers.Conv2D(256, (3, 3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.Conv2D(256, (3, 3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2, 2),
        layers.Dropout(0.40),

        # ── Dense head ──────────────────────────────────────────────────
        layers.Flatten(),
        layers.Dense(512, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.50),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.30),
        layers.Dense(num_classes, activation='softmax')
    ], name='EmotionCNN')

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

model = build_emotion_cnn(input_shape=(IMG_SIZE, IMG_SIZE, 1), num_classes=NUM_CLASSES)
model.summary()

## 5. Model Training

In [ ]:
model_save_path = str(MODEL_DIR / 'emotion_model.h5')

callbacks = [
    ModelCheckpoint(
        filepath=model_save_path,
        monitor='val_accuracy',
        save_best_only=True,
        mode='max',
        verbose=1
    ),
    EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-6,
        verbose=1
    )
]

history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)

print(f'\nModel saved to: {model_save_path}')

In [ ]:
# ── Training / validation curves ──────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['accuracy'],     label='Train Accuracy',     color='steelblue')
axes[0].plot(history.history['val_accuracy'], label='Validation Accuracy', color='coral')
axes[0].set_title('Model Accuracy', fontsize=13)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(history.history['loss'],     label='Train Loss',     color='steelblue')
axes[1].plot(history.history['val_loss'], label='Validation Loss', color='coral')
axes[1].set_title('Model Loss', fontsize=13)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=100, bbox_inches='tight')
plt.show()

## 6. Model Evaluation

In [ ]:
# ── Load best saved model ──────────────────────────────────────────────────
best_model = keras.models.load_model(model_save_path)

# ── Predictions ───────────────────────────────────────────────────────────
test_generator.reset()
y_pred_proba = best_model.predict(test_generator, verbose=1)
y_pred       = np.argmax(y_pred_proba, axis=1)
y_true       = test_generator.classes

# ── Core metrics ──────────────────────────────────────────────────────────
acc  = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred, average='weighted', zero_division=0)
rec  = recall_score(y_true, y_pred, average='weighted', zero_division=0)
f1   = f1_score(y_true, y_pred, average='weighted', zero_division=0)

print('=' * 45)
print(f'  Accuracy  : {acc:.4f}')
print(f'  Precision : {prec:.4f}  (weighted)')
print(f'  Recall    : {rec:.4f}  (weighted)')
print(f'  F1-Score  : {f1:.4f}  (weighted)')
print('=' * 45)

In [ ]:
# ── Classification report ─────────────────────────────────────────────────
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, zero_division=0))

In [ ]:
# ── Confusion Matrix ──────────────────────────────────────────────────────
cm = confusion_matrix(y_true, y_pred)
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

plt.figure(figsize=(10, 8))
sns.heatmap(
    cm_normalized, annot=True, fmt='.2f',
    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
    cmap='Blues', linewidths=0.5
)
plt.title('Normalised Confusion Matrix', fontsize=14)
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# ── ROC Curve (one-vs-rest) ────────────────────────────────────────────────
y_true_bin = label_binarize(y_true, classes=list(range(NUM_CLASSES)))

fpr, tpr, roc_auc = {}, {}, {}
for i in range(NUM_CLASSES):
    fpr[i], tpr[i], _ = roc_curve(y_true_bin[:, i], y_pred_proba[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

fpr['micro'], tpr['micro'], _ = roc_curve(y_true_bin.ravel(), y_pred_proba.ravel())
roc_auc['micro'] = auc(fpr['micro'], tpr['micro'])

plt.figure(figsize=(10, 7))
colors = cycle(sns.color_palette('tab10', NUM_CLASSES))
for i, color in zip(range(NUM_CLASSES), colors):
    plt.plot(fpr[i], tpr[i], color=color, lw=1.5,
             label=f'{CLASS_NAMES[i]} (AUC = {roc_auc[i]:.2f})')

plt.plot(fpr['micro'], tpr['micro'], color='black', lw=2, linestyle='--',
         label=f'Micro-average (AUC = {roc_auc["micro"]:.2f})')
plt.plot([0, 1], [0, 1], 'k:', lw=1)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves — One-vs-Rest (per emotion class)', fontsize=13)
plt.legend(loc='lower right', fontsize=9)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('roc_curves.png', dpi=100, bbox_inches='tight')
plt.show()

## 7. Save Model

In [ ]:
# Best model is already saved by ModelCheckpoint.
# Explicitly re-save to confirm path.
best_model.save(str(MODEL_DIR / 'emotion_model.h5'))
print('Model saved to:', MODEL_DIR / 'emotion_model.h5')

# Save class indices so prediction.py recovers the correct label mapping
indices_path = MODEL_DIR / 'class_indices.json'
with open(indices_path, 'w') as f:
    json.dump(train_generator.class_indices, f, indent=2)
print('Class indices saved to:', indices_path)

# Verify
loaded = keras.models.load_model(str(MODEL_DIR / 'emotion_model.h5'))
print('Reload OK. Input shape :', loaded.input_shape)
print('Reload OK. Output shape:', loaded.output_shape)